# PidginMed-ASR · Google MedASR Fine-tuning (CTC-safe)

Adapted from the uploaded **Google: Fine Tune with Hugging Face** notebook, with dataset paths, text normalization, speaker-independent shared splits, medical-term scoring and exports adapted from **02_google_medasr**.

**Critical fix:** LASR-CTC uses `labels != model.config.pad_token_id` to identify targets. Its collator must retain **tokenizer PAD IDs** in padded `labels`, **not `-100`**. Converting padding to `-100` can feed negative token IDs into CUDA CTC and cause an illegal memory access. The input `attention_mask` is also retained. Do not apply generic Wav2Vec2 CTC collator conventions here.

**Execution:** Run cells top-to-bottom in Google Colab after choosing GPU. Start with `DEBUG_MODE=True` for a short smoke test; after success switch to `False` and rerun from preprocessing. GPU errors can poison CUDA context—restart runtime before retrying. This notebook has *not* been run on your Drive/GPU.

**Sources**: [Google MedASR](https://huggingface.co/google/medasr), [LASR implementation](https://github.com/huggingface/transformers/blob/main/src/transformers/models/lasr/modeling_lasr.py). Model use subject to the model license.

## 1 · Install, mount Drive, environment

In [1]:
%pip install -q "transformers>=5.0,<6" "datasets>=3" accelerate evaluate jiwer librosa soundfile scikit-learn pandas tqdm huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Mounted at /content/drive
Fri Oct  9 12:21:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [ ]:
import os, re, ast, gc, json, hashlib, random
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import librosa, soundfile as sf
import torch
from datasets import Dataset
from jiwer import wer, cer
from transformers import (AutoProcessor, AutoModelForCTC, TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback, set_seed)
from tqdm.auto import tqdm

SEED = 42
MODEL_NAME = 'google/medasr'
BASE_DIR = Path('/content/drive/MyDrive/Brock_University/pidgindataset2026October')
METADATA_PATH = BASE_DIR / 'root' / 'Metadata.csv'
SPLIT_MANIFEST_PATH = BASE_DIR / 'shared_split_manifest.csv'
OUTPUT_ROOT = BASE_DIR / 'medasr_hf_clean'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEBUG_MODE = False                  # change to False for full training
DEBUG_TRAIN_SAMPLES = 32
DEBUG_VAL_SAMPLES = 16
RUN_TRAINING = True
RUN_TEST_EVALUATION = True          # only on a fully trained model, not debug
BATCH_SIZE = 4
GRAD_ACCUM = 4                     # effective batch = 16 on one GPU
EVAL_BATCH_SIZE = 4
LEARNING_RATE = 3e-5               # official guide starting rate; tune on validation
EPOCHS = 5
WARMUP_RATIO = 0.05
MAX_AUDIO_SECONDS = None           # None: do not truncate audio/text; choose duration policy explicitly
MIN_AUDIO_SECONDS = 0.6
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PyTorch:', torch.__version__)
import transformers
print('Transformers:', transformers.__version__)
print('Metadata:', METADATA_PATH)
print('Precision BF16:', BF16)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
PyTorch: 2.11.0+cu130
Transformers: 5.18.0
Metadata: /content/drive/MyDrive/Brock_University/pidgindataset2026October/root/Metadata.csv
Precision BF16: True


If the Hugging Face model requires access approval, accept the model terms and authenticate via the Colab secret `HF_TOKEN` (or run `huggingface_hub.notebook_login()`). Do not put your token in notebook source.

## 2 · Metadata, fixed paths and medical annotations

In [ ]:
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z\s']", ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def parse_terms(v):
    if isinstance(v, list):
        vals = v
    elif v is None or (not isinstance(v, (list, dict)) and pd.isna(v)):
        return []
    else:
        try:
            vals = ast.literal_eval(str(v))
            if not isinstance(vals, list): vals = [vals]
        except (SyntaxError, ValueError):
            vals = [p.strip() for p in str(v).split(',') if p.strip()]
    return list(dict.fromkeys(t for x in vals if (t := normalize_text(x))))

def fix_audio_path(s):
    # Only the filename: Speaker_14-1.wav -> Speaker14-1.wav; folder stays Speaker_14
    return re.sub(r'Speaker_(\d+)(?=-\d+\.wav$)', r'Speaker\1', str(s), flags=re.I)

def sample_id_for_row(r):
    for key in ('sample_id','utterance_id','id'):
        if key in r and pd.notna(r[key]) and str(r[key]).strip():
            return str(r[key]).strip()
    seed = '|'.join([str(r['speaker_id']), Path(str(r['audio_path'])).name, normalize_text(r['pidgin_transcription'])])
    return hashlib.sha1(seed.encode()).hexdigest()[:20]

def resolve_path(s):
    p = Path(str(s))
    return str(p if p.is_absolute() else BASE_DIR / p)

df = pd.read_csv(METADATA_PATH)
df.columns = df.columns.str.lower().str.strip()
required = ['audio_path', 'pidgin_transcription', 'speaker_id']
missing = set(required)-set(df.columns)
if missing: raise ValueError(f'Missing metadata columns: {missing}')
initial = len(df)
df = df.dropna(subset=required).copy()
df['audio_path'] = df['audio_path'].map(fix_audio_path)
df['audio'] = df['audio_path'].map(resolve_path)
df['reference'] = df['pidgin_transcription'].map(normalize_text)
df = df[df['reference'].str.len()>0].copy()
df = df[df['audio'].map(lambda p: Path(p).is_file())].copy()
if 'audio_length' in df:
    df['audio_length'] = pd.to_numeric(df['audio_length'], errors='coerce')
    # Recalculate missing audio duration using file headers
    miss = df['audio_length'].isna()
    def read_duration(p):
        try: return sf.info(p).duration
        except Exception: return np.nan
    df.loc[miss, 'audio_length'] = df.loc[miss, 'audio'].map(read_duration)
else:
    df['audio_length'] = df['audio'].map(lambda p: sf.info(p).duration)
df = df[df['audio_length'].ge(MIN_AUDIO_SECONDS) & df['audio_length'].notna()].copy()
if MAX_AUDIO_SECONDS is not None:
    df = df[df['audio_length'].le(MAX_AUDIO_SECONDS)].copy()  # excludes, never truncates
med_col = next((c for c in ('medical_terms_pidgen','medical_terms_clean') if c in df), None)
df['medical_terms_pidgen'] = df[med_col].map(parse_terms) if med_col else [[] for _ in range(len(df))]
df['medical_annotation_present'] = (df[med_col].notna() & ~df[med_col].astype(str).str.lower().str.strip().isin(['','nan','none','null'])) if med_col else False
df['sample_id'] = df.apply(sample_id_for_row, axis=1).astype(str)
if df['sample_id'].duplicated().any(): raise ValueError('Duplicate sample IDs; resolve before splitting')
df['speaker_id'] = df['speaker_id'].astype(str)
df = df.reset_index(drop=True)
print('Original:', initial, 'Valid:', len(df), 'Speakers:', df.speaker_id.nunique())
print('Hours:', round(df.audio_length.sum()/3600, 2))
print('Medical empty lists preserved:', sum(df.medical_terms_pidgen.map(len)==0))

/tmp/ipykernel_12881/3683606427.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[miss, 'audio_length'] = df.loc[miss, 'audio'].map(read_duration)


Original: 10218 Valid: 10218 Speakers: 21
Hours: 78.98
Medical empty lists preserved: 283


## 3 · Reuse shared speaker-disjoint manifest (no silent resplitting)

In [ ]:
if SPLIT_MANIFEST_PATH.exists():
    manifest = pd.read_csv(SPLIT_MANIFEST_PATH, dtype={'sample_id':str,'speaker_id':str})
    if not {'sample_id','speaker_id','split'}.issubset(manifest):
        raise ValueError('Existing split manifest requires sample_id, speaker_id and split')
    if manifest.sample_id.duplicated().any(): raise ValueError('Duplicate manifest sample IDs')
    if not set(df.sample_id).issubset(set(manifest.sample_id)):
        raise ValueError('Current sample IDs not covered by shared manifest; reconcile IDs, do not silently rebuild')
    df = df.merge(manifest[['sample_id','speaker_id','split']], on='sample_id', suffixes=('','_manifest'), validate='one_to_one', how='left')
    if not (df.speaker_id == df.speaker_id_manifest).all(): raise ValueError('Speaker mismatch with manifest')
    df = df.drop(columns='speaker_id_manifest')
    print('Loaded existing manifest:', SPLIT_MANIFEST_PATH)
else:
    from sklearn.model_selection import train_test_split
    speakers = sorted(df.speaker_id.unique())
    train_val, test_spk = train_test_split(speakers, test_size=.20, random_state=SEED)
    train_spk, val_spk = train_test_split(train_val, test_size=.15, random_state=SEED)
    mapping = {**{s:'train' for s in train_spk}, **{s:'validation' for s in val_spk}, **{s:'test' for s in test_spk}}
    df['split'] = df.speaker_id.map(mapping)
    cols = ['sample_id','speaker_id','split','audio_path','pidgin_transcription']
    df[cols].to_csv(SPLIT_MANIFEST_PATH, index=False)
    print('Created shared manifest:', SPLIT_MANIFEST_PATH)
train_df = df[df.split=='train'].reset_index(drop=True)
val_df = df[df.split=='validation'].reset_index(drop=True)
test_df = df[df.split=='test'].reset_index(drop=True)
sets = [set(x.speaker_id) for x in (train_df, val_df, test_df)]
assert all(sets[i].isdisjoint(sets[j]) for i in range(3) for j in range(i+1,3))
assert all(len(x)>0 for x in (train_df,val_df,test_df))
print(df.groupby('split').agg(samples=('sample_id','size'),speakers=('speaker_id','nunique'),hours=('audio_length',lambda x:round(x.sum()/3600,2))))
if DEBUG_MODE:
    train_df = train_df.head(DEBUG_TRAIN_SAMPLES).copy()
    val_df = val_df.head(DEBUG_VAL_SAMPLES).copy()
    print('DEBUG ONLY:', len(train_df), 'train /',len(val_df),'validation')

Loaded existing manifest: /content/drive/MyDrive/Brock_University/pidgindataset2026October/shared_split_manifest.csv
            samples  speakers  hours
split                               
test           2245         5  17.76
train          6474        13  48.96
validation     1499         3  12.25


In [ ]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 22.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 4 · Prepare 16 kHz audio features and LASR CTC labels

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
config_model = AutoModelForCTC.from_pretrained(MODEL_NAME)
PAD_ID = int(config_model.config.pad_token_id)
VOCAB_SIZE = int(config_model.config.vocab_size)
TOKENIZER_PAD = int(processor.tokenizer.pad_token_id)
assert PAD_ID == TOKENIZER_PAD, f'PAD mismatch: model={PAD_ID} tokenizer={TOKENIZER_PAD}'
print('LASR PAD/blank:',PAD_ID, 'vocab:',VOCAB_SIZE)
del config_model

def encode_one(row):
    audio, sr = librosa.load(row['audio'], sr=16000, mono=True)
    if not np.isfinite(audio).all(): raise ValueError(f'Nonfinite waveform: {row["audio"]}')
    features = processor(audio, sampling_rate=16000).input_features[0]
    tokens = processor.tokenizer(row['reference']).input_ids  # follows Google's official guide
    return {'input_features':features, 'labels':tokens}

def encode_split(frame, name):
    ds = Dataset.from_pandas(frame[['audio','reference']].copy(), preserve_index=False)
    return ds.map(encode_one, remove_columns=ds.column_names, desc='Prepare '+name)

encoded_train = encode_split(train_df,'train')
encoded_val = encode_split(val_df,'validation')
print(encoded_train, encoded_val)

Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

LASR PAD/blank: 0 vocab: 512


Prepare train:   0%|          | 0/6474 [00:00<?, ? examples/s]

Prepare validation:   0%|          | 0/1499 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 6474
}) Dataset({
    features: ['input_features', 'labels'],
    num_rows: 1499
})


## 5 · **Critical LASR-safe collator** and CTC integrity checks

In [ ]:
@dataclass
class LASRDataCollator:
    processor: object
    pad_id: int
    def __call__(self, features):
        audio = self.processor.feature_extractor.pad(
            [{'input_features':f['input_features']} for f in features],
            padding=True, return_attention_mask=True, return_tensors='pt')
        target = self.processor.tokenizer.pad(
            [{'input_ids':f['labels']} for f in features],
            padding=True, return_tensors='pt')
        # CRITICAL: LASR expects PAD_ID, not -100, as label padding
        audio['labels'] = target['input_ids']
        if audio['labels'].min().item()<0: raise ValueError('Negative CTC target ID!')
        if (audio['labels']>=VOCAB_SIZE).any(): raise ValueError('CTC target ID exceeds vocabulary')
        return audio

collator = LASRDataCollator(processor, PAD_ID)

def audit_encoded(ds, label):
    problems=[]
    for idx,x in enumerate(ds):
        y=np.asarray(x['labels'],dtype=np.int64)
        a=np.asarray(x['input_features'])
        if y.ndim != 1 or len(y)==0: problems.append((idx,'empty/non-1D labels'))
        if ((y<0)|(y>=VOCAB_SIZE)).any(): problems.append((idx,'invalid token ID'))
        if (y==PAD_ID).any(): problems.append((idx,'blank/PAD in unpadded labels'))
        if not np.isfinite(a).all(): problems.append((idx,'invalid audio features'))
    print(label, 'checked',len(ds),'problems',len(problems), problems[:15])
    if problems: raise ValueError(f'Invalid {label} examples: {problems[:10]}')
audit_encoded(encoded_train, 'train')
audit_encoded(encoded_val, 'validation')
smoke = collator([encoded_train[i] for i in range(min(2,len(encoded_train)))])
print({k:tuple(v.shape) for k,v in smoke.items()})
assert smoke['labels'].min().item() >= 0
assert 'attention_mask' in smoke
print('PASSED: LASR labels use PAD_ID and attention_mask is included')

train checked 6474 problems 0 []
validation checked 1499 problems 0 []
{'input_features': (2, 3794, 128), 'attention_mask': (2, 3794), 'labels': (2, 151)}
PASSED: LASR labels use PAD_ID and attention_mask is included


## 6 · Small forward/backward smoke test before training

In [ ]:
def smoke_backward(batch):
    m=AutoModelForCTC.from_pretrained(MODEL_NAME).to(DEVICE).train()
    batch={k:v.to(DEVICE) for k,v in batch.items()}
    m.zero_grad(set_to_none=True)
    out=m(**batch)
    print('Initial CTC loss:',float(out.loss.detach().cpu()))
    if not torch.isfinite(out.loss): raise ValueError('Nonfinite CTC loss')
    out.loss.backward()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    del m, out
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Forward and backward: PASSED')
smoke_backward(smoke)

Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

Initial CTC loss: 2.527045726776123
Forward and backward: PASSED


## 7 · WER/CER evaluation and Trainer configuration

In [ ]:
def compute_metrics(eval_pred):
    logits=eval_pred.predictions
    if isinstance(logits,tuple): logits=logits[0]
    predictions=np.argmax(logits,axis=-1)
    texts=[normalize_text(x) for x in processor.batch_decode(predictions,skip_special_tokens=True)]
    labels=np.asarray(eval_pred.label_ids).copy()
    labels[labels==-100]=PAD_ID  # defensive: trainer should not use -100 for LASR labels
    refs=[normalize_text(x) for x in processor.tokenizer.batch_decode(labels,skip_special_tokens=True)]
    return {'wer':wer(refs,texts), 'cer':cer(refs,texts)}

class EpochCSV(TrainerCallback):
    def __init__(self,path): self.path=Path(path);self.entries=[]
    def on_evaluate(self,args,state,control,metrics=None,**kwargs):
        record={'epoch':state.epoch,'step':state.global_step,**(metrics or {})}
        self.entries.append(record)
        pd.DataFrame(self.entries).to_csv(self.path,index=False)

model=AutoModelForCTC.from_pretrained(MODEL_NAME)
RUN_DIR=OUTPUT_ROOT / ('debug_run' if DEBUG_MODE else 'full_run')
RUN_DIR.mkdir(parents=True,exist_ok=True)
args=TrainingArguments(
    output_dir=str(RUN_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=1 if DEBUG_MODE else EPOCHS,
    max_steps=5 if DEBUG_MODE else -1,
    learning_rate=LEARNING_RATE,
    warmup_steps=300,
    weight_decay=.01,

    optim='adamw_torch',
    bf16=BF16,
    fp16=False,
    gradient_checkpointing=False,
    eval_strategy='epoch',
    save_strategy='no' if DEBUG_MODE else 'epoch',
    save_total_limit=2,
    load_best_model_at_end=not DEBUG_MODE,
    metric_for_best_model='wer' if not DEBUG_MODE else None,
    greater_is_better=False,
    logging_steps=1 if DEBUG_MODE else 10,
    logging_first_step=True,
    dataloader_num_workers=0 if DEBUG_MODE else 2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
)
trainer=Trainer(
    model=model,args=args,train_dataset=encoded_train,eval_dataset=encoded_val,
    data_collator=collator,processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[EpochCSV(RUN_DIR/'epoch_metrics.csv')]+([] if DEBUG_MODE else [EarlyStoppingCallback(early_stopping_patience=3)])
)
print('Effective batch size:',BATCH_SIZE*GRAD_ACCUM)

Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

Effective batch size: 16


## 8 · Fine-tune and persist best model

In [ ]:
if RUN_TRAINING:
    result=trainer.train()
    val_metrics=trainer.evaluate()
    print('Validation:',val_metrics)
    if not DEBUG_MODE:
        selected_dir=OUTPUT_ROOT/'selected_model'
        trainer.save_model(str(selected_dir))
        processor.save_pretrained(str(selected_dir))
        (OUTPUT_ROOT/'validation_metrics.json').write_text(json.dumps(val_metrics,indent=2,default=float))
        print('Saved best selected model:',selected_dir)
    else:
        print('DEBUG mode: no final model exported; rerun in full mode for a benchmark.')

Epoch,Training Loss,Validation Loss,Wer,Cer
1,8.078651,1.500665,0.444152,0.210957
2,5.522131,1.021947,0.337357,0.159582
3,5.488920,0.918788,0.299003,0.145003
4,5.865842,0.880969,0.287128,0.139913
5,5.294084,0.870900,0.284534,0.138785


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Wer,Cer
5.294084,0.870900,5,0.284534,0.138785


Validation: {'eval_loss': 0.8709003329277039, 'eval_wer': 0.28453350288163753, 'eval_cer': 0.13878456844993575}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best selected model: /content/drive/MyDrive/Brock_University/pidgindataset2026October/medasr_hf_clean/selected_model


## 9 · Held-out test: aligned predictions, WER, CER, medical-term metrics & corpus MWER

Test is evaluated **once after full training**. The lexicon-based MWER below reproduces the prior notebook's *term-presence approximation*, not word-aligned clinical MWER. It evaluates only utterances with medical terms in the Pidgin reference. Empty medical-term lists are kept during ASR training. Annotations that are missing rather than verified-empty should not be considered reliable negative medical-term examples.

In [ ]:
def present_terms(text,terms):
    padded=' '+normalize_text(text)+' '
    return set(t for t in (normalize_text(x) for x in terms) if t and (' '+t+' ') in padded)

def evaluate_predictions(frame):
    frame=frame.copy()
    frame['ref_terms']=frame.apply(lambda r:sorted(present_terms(r.reference,r.medical_terms_pidgen)),axis=1)
    frame['hyp_terms']=frame.apply(lambda r:sorted(present_terms(r.prediction,r.medical_terms_pidgen)),axis=1)
    tp=fp=fn=0; ref_med=[]; hyp_med=[]; exact=0; med_n=0
    for _,r in frame.iterrows():
        ref=set(r.ref_terms); hyp=set(r.hyp_terms)
        if not ref: continue
        med_n+=1; tp+=len(ref&hyp);fp+=len(hyp-ref);fn+=len(ref-hyp);exact+=int(ref==hyp)
        ref_med.append(' '.join(sorted(ref))); hyp_med.append(' '.join(sorted(hyp)))
    prec=tp/(tp+fp) if tp+fp else 0.;rec=tp/(tp+fn) if tp+fn else 0.
    metrics={'num_samples':len(frame), 'wer':wer(frame.reference.tolist(),frame.prediction.tolist()),
             'cer':cer(frame.reference.tolist(),frame.prediction.tolist()),
             'mwer':wer(ref_med,hyp_med) if ref_med else float('nan'),
             'mwer_evaluated_samples':med_n, 'medical_precision':prec, 'medical_recall':rec,
             'medical_f1':(2*prec*rec/(prec+rec)) if prec+rec else 0.,
             'medical_accuracy':exact/med_n if med_n else float('nan'),
             'medical_tp':tp,'medical_fp':fp,'medical_fn':fn}
    return frame, metrics

def predict_test(frame, model, batch_size=2):
    model.eval(); dev=next(model.parameters()).device; predictions=[]
    for start in tqdm(range(0,len(frame),batch_size),desc='Held-out test'):
        sub=frame.iloc[start:start+batch_size]
        waves=[librosa.load(path,sr=16000,mono=True)[0] for path in sub.audio]
        inp=processor(waves,sampling_rate=16000,padding=True,return_tensors='pt',return_attention_mask=True)
        inp={k:v.to(dev) for k,v in inp.items()}
        with torch.inference_mode():
            tokens=model.generate(**inp)
        predictions.extend(normalize_text(x) for x in processor.batch_decode(tokens,skip_special_tokens=True))
    result=frame.copy(); result['prediction']=predictions
    return result

if RUN_TEST_EVALUATION and RUN_TRAINING and not DEBUG_MODE:
    test_predictions=predict_test(test_df,trainer.model)
    test_predictions, test_metrics=evaluate_predictions(test_predictions)
    test_predictions.to_csv(OUTPUT_ROOT/'test_predictions.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame([test_metrics]).to_csv(OUTPUT_ROOT/'test_metrics.csv',index=False)
    (OUTPUT_ROOT/'test_metrics.json').write_text(json.dumps(test_metrics,indent=2))
    print(pd.Series(test_metrics).to_string())
else:
    print('Held-out test skipped (debug mode, or disabled).')

Held-out test:   0%|          | 0/1123 [00:00<?, ?it/s]

num_samples                2245.000000
wer                           0.290453
cer                           0.144048
mwer                          0.418392
mwer_evaluated_samples     2217.000000
medical_precision             0.990497
medical_recall                0.647640
medical_f1                    0.783189
medical_accuracy              0.116373
medical_tp                11153.000000
medical_fp                  107.000000
medical_fn                 6068.000000


## 10 · Reproducibility and run summary

In [ ]:
record={'model':MODEL_NAME,'seed':SEED,'debug':DEBUG_MODE,
        'train_examples':len(train_df),'validation_examples':len(val_df),
        'test_examples':len(test_df),'train_batch':BATCH_SIZE,'accumulation':GRAD_ACCUM,
        'learning_rate':LEARNING_RATE,'epochs':EPOCHS,'bf16':BF16,
        'metadata_path':str(METADATA_PATH),'manifest_path':str(SPLIT_MANIFEST_PATH),
        'torch_version':torch.__version__,'transformers_version':transformers.__version__,
        'critical_lasr_fix':'label padding uses model PAD_ID, never -100'}
(OUTPUT_ROOT/('debug_config.json' if DEBUG_MODE else 'run_config.json')).write_text(json.dumps(record,indent=2))
print('Wrote configuration to:',OUTPUT_ROOT)
print('Complete. Debug mode:',DEBUG_MODE)

Wrote configuration to: /content/drive/MyDrive/Brock_University/pidgindataset2026October/medasr_hf_clean
Complete. Debug mode: False
